In [2]:
import sys
import os
if 'utils' in sys.modules:
    del sys.modules['utils']
sys.path.append('../src')

import pandas as pd
import numpy as np
import networkx as nx
from utils import load_dataset, train_test_split_temporal, EDGE_FEATURES, WINDOW_HOURS

# Carica il subset
df = load_dataset('../data/ton_iot_20pct.csv')
print(f"Dataset caricato: {len(df):,} righe")
print(f"Periodo: {df['Timestamp'].min()} → {df['Timestamp'].max()}")
print(f"Finestra temporale: {WINDOW_HOURS} ore")

Dataset caricato: 1,070,352 righe
Periodo: 2019-04-02 19:58:04 → 2019-04-30 00:45:55
Finestra temporale: 2 ore


Creazioen finestre temporali

In [3]:
# Assegna ogni flusso alla sua finestra temporale
df['window'] = df['Timestamp'].dt.floor(f'{WINDOW_HOURS}h')

# Conta quanti flussi ci sono per finestra
window_counts = df.groupby('window').size()

print(f"Finestre totali: {len(window_counts)}")
print(f"Finestre non vuote: {(window_counts > 0).sum()}")
print(f"Media flussi per finestra: {window_counts.mean():.0f}")
print(f"Min flussi per finestra:   {window_counts.min()}")
print(f"Max flussi per finestra:   {window_counts.max()}")

# Quante finestre hanno almeno 10 flussi (utili per costruire grafi)
print(f"\nFinestre con >10 flussi: {(window_counts > 10).sum()}")
print(f"Finestre con >100 flussi: {(window_counts > 100).sum()}")

Finestre totali: 78
Finestre non vuote: 78
Media flussi per finestra: 13722
Min flussi per finestra:   1
Max flussi per finestra:   241775

Finestre con >10 flussi: 73
Finestre con >100 flussi: 57


Funzione di costruzione grafo per una finestra

In [4]:
def build_graph(window_df, edge_types):
    """
    Costruisce un grafo NetworkX da un dataframe di flussi.
    
    edge_types: lista di tipi di arco da includere
                es. ['comm'] oppure ['comm', 'context']
    
    Ritorna un grafo NetworkX con:
    - nodi = IP unici
    - archi = flussi, con features e label
    """
    G = nx.MultiDiGraph()  # diretto e multi-arco
    
    # Pulizia valori infiniti e NaN
    window_df = window_df.replace([np.inf, -np.inf], np.nan).fillna(0)
    
    for _, row in window_df.iterrows():
        src = row['Src IP']
        dst = row['Dst IP']
        label = int(row['Label'])
        
        # Aggiungi nodi se non esistono
        if src not in G:
            G.add_node(src)
        if dst not in G:
            G.add_node(dst)
        
        # Aggiungi archi per ogni tipo richiesto
        for etype in edge_types:
            features = {}
            for feat in EDGE_FEATURES[etype]:
                if feat in row.index:
                    features[feat] = row[feat]
            
            G.add_edge(src, dst,
                      edge_type=etype,
                      label=label,
                      **features)
    
    return G

# Test su una finestra
test_window = df[df['window'] == df['window'].unique()[50]]
G_test = build_graph(test_window, edge_types=['comm'])

print(f"Finestra di test: {df['window'].unique()[50]}")
print(f"Flussi nella finestra: {len(test_window)}")
print(f"Nodi nel grafo: {G_test.number_of_nodes()}")
print(f"Archi nel grafo: {G_test.number_of_edges()}")
print(f"Archi anomali: {sum(1 for u,v,d in G_test.edges(data=True) if d['label']==1)}")

Finestra di test: 2019-04-26 04:00:00
Flussi nella finestra: 1614
Nodi nel grafo: 1470
Archi nel grafo: 1614
Archi anomali: 0


Costruzioe di tutti i grafi per tutte le configurazioni

In [5]:
import torch
import os

# Filtra finestre con almeno 100 flussi
valid_windows = window_counts[window_counts >= 100].index
print(f"Finestre valide: {len(valid_windows)}")

# Costruisci e salva grafi per ogni configurazione
configs = {
    'A': ['comm'],
    'B': ['comm', 'context'],
    'C': ['comm', 'knowledge'],
    'D': ['comm', 'context', 'knowledge']
}

all_graphs = {}  # config -> lista di grafi

for config_name, edge_types in configs.items():
    print(f"\nCostruendo CONFIG_{config_name}...")
    graphs = []
    
    for window in valid_windows:
        window_df = df[df['window'] == window].copy()
        G = build_graph(window_df, edge_types)
        graphs.append({
            'window': window,
            'graph': G,
            'n_nodes': G.number_of_nodes(),
            'n_edges': G.number_of_edges(),
            'n_anomalies': sum(1 for u,v,d in G.edges(data=True) if d['label']==1)
        })
    
    all_graphs[config_name] = graphs
    print(f"  Grafi costruiti: {len(graphs)}")
    print(f"  Nodi medi: {np.mean([g['n_nodes'] for g in graphs]):.0f}")
    print(f"  Archi medi: {np.mean([g['n_edges'] for g in graphs]):.0f}")
    print(f"  Anomalie medie per grafo: {np.mean([g['n_anomalies'] for g in graphs]):.0f}")

print("\nDone.")

Finestre valide: 57

Costruendo CONFIG_A...
  Grafi costruiti: 57
  Nodi medi: 1675
  Archi medi: 18771
  Anomalie medie per grafo: 9953

Costruendo CONFIG_B...
  Grafi costruiti: 57
  Nodi medi: 1675
  Archi medi: 37543
  Anomalie medie per grafo: 19905

Costruendo CONFIG_C...
  Grafi costruiti: 57
  Nodi medi: 1675
  Archi medi: 37543
  Anomalie medie per grafo: 19905

Costruendo CONFIG_D...
  Grafi costruiti: 57
  Nodi medi: 1675
  Archi medi: 56314
  Anomalie medie per grafo: 29858

Done.


Split Train/Test

In [6]:
# Split temporale: primi 70% dei grafi per train, ultimi 30% per test
def split_graphs(graphs, test_ratio=0.3):
    split_idx = int(len(graphs) * (1 - test_ratio))
    return graphs[:split_idx], graphs[split_idx:]

splits = {}
for config_name, graphs in all_graphs.items():
    train, test = split_graphs(graphs)
    splits[config_name] = {'train': train, 'test': test}
    print(f"CONFIG_{config_name}: {len(train)} train, {len(test)} test grafi")

CONFIG_A: 39 train, 18 test grafi
CONFIG_B: 39 train, 18 test grafi
CONFIG_C: 39 train, 18 test grafi
CONFIG_D: 39 train, 18 test grafi


Prova - Benchmark Velocità

In [7]:
import time

# Prendi la finestra più grande
biggest_window = window_counts.idxmax()
biggest_df = df[df['window'] == biggest_window].copy()

print(f"Finestra più grande: {biggest_window}")
print(f"Flussi: {len(biggest_df):,}")

# Misura tempo per CONFIG_D (caso peggiore - tutti e 3 i tipi di arco)
start = time.time()
G_big = build_graph(biggest_df, edge_types=['comm', 'context', 'knowledge'])
elapsed = time.time() - start

print(f"\nTempo costruzione grafo: {elapsed:.1f} secondi")
print(f"Tempo stimato per tutti i grafi CONFIG_D: {elapsed * 57:.0f} secondi ({elapsed * 57 / 60:.1f} minuti)")

Finestra più grande: 2019-04-27 16:00:00
Flussi: 241,775

Tempo costruzione grafo: 17.9 secondi
Tempo stimato per tutti i grafi CONFIG_D: 1021 secondi (17.0 minuti)


Verifica PyTorch Geometric

In [8]:
import torch
import torch_geometric
print(f"PyTorch: {torch.__version__}")
print(f"PyTorch Geometric: {torch_geometric.__version__}")
print(f"MPS disponibile (Apple Silicon): {torch.backends.mps.is_available()}")

PyTorch: 2.9.1
PyTorch Geometric: 2.7.0
MPS disponibile (Apple Silicon): True


Conversioe da NetworkX a PyTorch Geometric

In [9]:
from torch_geometric.data import HeteroData
from sklearn.preprocessing import MinMaxScaler

def graph_to_heterodata(graph_dict, edge_types):
    """
    Converte un grafo NetworkX in formato HeteroData di PyTorch Geometric.
    """
    G = graph_dict['graph']
    
    # Mappa IP -> indice intero
    nodes = list(G.nodes())
    node_idx = {ip: i for i, ip in enumerate(nodes)}
    n_nodes = len(nodes)
    
    # Feature dei nodi: in-degree e out-degree
    in_deg  = dict(G.in_degree())
    out_deg = dict(G.out_degree())
    
    node_features = torch.tensor(
        [[in_deg[n], out_deg[n]] for n in nodes],
        dtype=torch.float
    )
    
    data = HeteroData()
    data['node'].x = node_features
    data['node'].num_nodes = n_nodes
    
    # Archi per ogni tipo
    for etype in edge_types:
        src_list, dst_list, feat_list, label_list = [], [], [], []
        feat_cols = EDGE_FEATURES[etype]
        
        for u, v, d in G.edges(data=True):
            if d.get('edge_type') != etype:
                continue
            src_list.append(node_idx[u])
            dst_list.append(node_idx[v])
            label_list.append(d.get('label', 0))
            feat_list.append([d.get(f, 0.0) for f in feat_cols])
        
        if len(src_list) == 0:
            continue
            
        edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
        edge_attr  = torch.tensor(feat_list, dtype=torch.float)
        edge_label = torch.tensor(label_list, dtype=torch.long)
        
        # Normalizza features
        scaler = MinMaxScaler()
        edge_attr = torch.tensor(
            scaler.fit_transform(edge_attr.numpy()),
            dtype=torch.float
        )
        
        data['node', etype, 'node'].edge_index = edge_index
        data['node', etype, 'node'].edge_attr  = edge_attr
        data['node', etype, 'node'].edge_label = edge_label
    
    return data

# Test su un grafo
sample = all_graphs['D'][40]
edge_types = ['comm', 'context', 'knowledge']
hetero_sample = graph_to_heterodata(sample, edge_types)
print(hetero_sample)

HeteroData(
  node={
    x=[415, 2],
    num_nodes=415,
  },
  (node, comm, node)={
    edge_index=[2, 19078],
    edge_attr=[19078, 8],
    edge_label=[19078],
  },
  (node, context, node)={
    edge_index=[2, 19078],
    edge_attr=[19078, 4],
    edge_label=[19078],
  },
  (node, knowledge, node)={
    edge_index=[2, 19078],
    edge_attr=[19078, 8],
    edge_label=[19078],
  }
)


Conversione e Salvataggio di tutti i Grafi

In [10]:
import torch
import os

CONFIGS = {
    'A': ['comm'],
    'B': ['comm', 'context'],
    'C': ['comm', 'knowledge'],
    'D': ['comm', 'context', 'knowledge']
}

for config_name, edge_types in CONFIGS.items():
    print(f"\nConvertendo CONFIG_{config_name}...")
    
    save_dir = f'../outputs/graphs/CONFIG_{config_name}'
    os.makedirs(save_dir, exist_ok=True)
    
    for split_name, graphs in splits[config_name].items():
        split_dir = f'{save_dir}/{split_name}'
        os.makedirs(split_dir, exist_ok=True)
        
        for i, graph_dict in enumerate(graphs):
            hetero = graph_to_heterodata(graph_dict, edge_types)
            torch.save(hetero, f'{split_dir}/graph_{i:03d}.pt')
        
        print(f"  {split_name}: {len(graphs)} grafi salvati in {split_dir}")

print("\nDone.")


Convertendo CONFIG_A...
  train: 39 grafi salvati in ../outputs/graphs/CONFIG_A/train
  test: 18 grafi salvati in ../outputs/graphs/CONFIG_A/test

Convertendo CONFIG_B...
  train: 39 grafi salvati in ../outputs/graphs/CONFIG_B/train
  test: 18 grafi salvati in ../outputs/graphs/CONFIG_B/test

Convertendo CONFIG_C...
  train: 39 grafi salvati in ../outputs/graphs/CONFIG_C/train
  test: 18 grafi salvati in ../outputs/graphs/CONFIG_C/test

Convertendo CONFIG_D...
  train: 39 grafi salvati in ../outputs/graphs/CONFIG_D/train
  test: 18 grafi salvati in ../outputs/graphs/CONFIG_D/test

Done.


Rigeneriamo i grafi con community label

In [11]:
from networkx.algorithms.community import label_propagation_communities
from utils import run_lpa, add_community_labels

for config_name, edge_types in CONFIGS.items():
    print(f"\nRigenerando CONFIG_{config_name} con community labels...")
    
    for split_name, graphs in splits[config_name].items():
        split_dir = f'../outputs/graphs/CONFIG_{config_name}/{split_name}'
        
        for i, graph_dict in enumerate(graphs):
            G = graph_dict['graph']
            node_list = list(G.nodes())
            
            # Community detection
            community_map = run_lpa(G)
            
            # Carica heterodata esistente
            hetero = torch.load(
                f'{split_dir}/graph_{i:03d}.pt',
                weights_only=False
            )
            
            # Aggiungi community labels
            hetero = add_community_labels(hetero, community_map, node_list)
            
            # Salva
            torch.save(hetero, f'{split_dir}/graph_{i:03d}.pt')
        
        print(f"  {split_name}: {len(graphs)} grafi aggiornati")

print("\nDone.")


Rigenerando CONFIG_A con community labels...
  train: 39 grafi aggiornati
  test: 18 grafi aggiornati

Rigenerando CONFIG_B con community labels...
  train: 39 grafi aggiornati
  test: 18 grafi aggiornati

Rigenerando CONFIG_C con community labels...
  train: 39 grafi aggiornati
  test: 18 grafi aggiornati

Rigenerando CONFIG_D con community labels...
  train: 39 grafi aggiornati
  test: 18 grafi aggiornati

Done.
